In [ ]:
import itertools
import json

import numpy as np

from theia.data_loading import load_bakom_ukw_transmitters
from theia.distance import line_of_sight_distance
from theia.grids import LatLonHeightGrid
from theia.test_data import get_uetliberg_radar
from theia.types import Receiver, Transmitter


def save_pcl_test_data(
    path: str,
    data: tuple[
        list[Receiver],
        list[Transmitter],
        LatLonHeightGrid,
        list[np.ndarray],
    ]
    | None = None,
):
    if data is None:
        p = get_uetliberg_radar(integration_time=0.1).receiver.point

        transmitters = load_bakom_ukw_transmitters()
        txs = list(
            filter(
                lambda t: (
                    line_of_sight_distance(*p.as_tuple(), *t.point.as_tuple()) < 50_000
                ),
                transmitters,
            )
        )[:3]

        grid = LatLonHeightGrid(
            lat_start=46.9973,
            lat_stop=47.5282,
            lat_res=(47.5282 - 46.9973) / 4.0,
            lon_start=7.8635,
            lon_stop=9.1956,
            lon_res=(9.1956 - 7.8635) / 4.0,
            height_start=1000.0,
            height_stop=1000.0,
            height_res=1000.0,
        )

        rxs: list[Receiver] = []
        true_values: list[list[float]] = []
        for tx in txs:
            rx = get_uetliberg_radar(
                integration_time=0.1,
                tx_bandwidth=tx.bandwidth,
            ).receiver
            rxs.append(rx)
            true_values.append(np.full(grid.n_points, np.nan).tolist())
    else:
        rxs, txs, grid, true_values = data
        true_values = [values.tolist() for values in true_values]

    configs = []
    for rx, tx, values in zip(rxs, txs, true_values, strict=True):
        config = {
            "rx": rx.model_dump(mode="json"),
            "tx": tx.model_dump(mode="json"),
            "grid": grid.model_dump(mode="json"),
            "true_values": values,
        }
        configs.append(config)

    with open(path, "w") as file:
        json.dump(configs, file)


def load_pcl_test_data(
    path: str,
) -> tuple[list[Receiver], list[Transmitter], list[LatLonHeightGrid], list[np.ndarray]]:
    with open("file.json", "r") as file:
        configs_loaded = json.load(file)
    rxs_loaded = [Receiver.model_validate(entry["rx"]) for entry in configs_loaded]
    txs_loaded = [Transmitter.model_validate(entry["tx"]) for entry in configs_loaded]
    grids = [LatLonHeightGrid.model_validate(entry["grid"]) for entry in configs_loaded]
    true_values = [np.array(entry["true_values"]) for entry in configs_loaded]
    return rxs_loaded, txs_loaded, grids, true_values

In [ ]:
# from tqdm import tqdm

# from theia.openburst_client import OpenburstClient

# save_pcl_test_data("file.json")
# rxs, txs, grids, _ = load_pcl_test_data("file.json")

# client = OpenburstClient("localhost")

# true_values = []
# for rx, tx, grid in tqdm(zip(rxs, txs, grids, strict=True)):
#     result = client.calculate_min_det_rcs_coverage(
#         rx,
#         tx,
#         grid,
#     )[0]
#     true_values.append(result)

# save_pcl_test_data("file.json", data=(rxs, txs, grid, true_values))

In [ ]:
rxs, txs, grids, values_loaded = load_pcl_test_data("file.json")

In [ ]:
from theia.detection.pcl import PclDetector

rx, tx, grid, true_values = next(zip(rxs, txs, grids, values_loaded, strict=True))
detector = PclDetector()
result_calc = detector.minimum_detectable_rcs_vector(rx, tx, grid.points)
result_calc = result_calc.reshape(grid.n_points)
np.nan_to_num(result_calc, copy=False, nan=-1)

In [ ]:
a = grid.points

In [ ]:
a[3, :]

In [ ]:
detector

In [ ]:
detector.minimum_detectable_rcs_vector(rx, tx, grid.points[3, :].reshape((1, -1)))

In [ ]:
import numpy as np

from theia.util import to_dB


error = np.abs((true_values - np.clip(result_calc, -1, 150.0)))
i = np.argmax(error)

In [ ]:
i

In [ ]:
true_values.flatten()[3], result_calc.flatten()[3]

In [ ]:
np.log10(true_values.flatten()[i]) * 10.0 - np.log10(result_calc.flatten()[i]) * 10.0

In [ ]:
true_values.flatten()[i], result_calc.flatten()[i]

In [ ]:
np.where(result_calc == -1)

In [ ]:
true_values[np.where(result_calc == -1)]

In [ ]:
# import folium

# from theia.mapping import RadarMap
# from theia.types import MonostaticRadarMeasurementModel, Radar

# radars = {
#     str(tx.id): Radar(
#         transmitter=tx,
#         receiver=rxs[0],
#         error_model=MonostaticRadarMeasurementModel(),
#     )
#     for tx in txs
# }
# radars["Üetliberg"] = radar

# map = RadarMap(
#     radars=radars,
# ).to_map()
# for point in grid.points:
#     folium.CircleMarker(
#         location=(point[0], point[1]),
#         fill_color="red",
#         color="red",
#     ).add_to(map)
# map.location = (rx.lat, rx.lon)
# map

In [ ]:
from theia.util import to_dB


to_dB(tx.erp)